# 🌸 Hasaki RAG Chatbot - Google Colab Deployment

Notebook này giúp bạn deploy RAG Chatbot lên Google Colab với Ngrok tunnel.

## 📋 Các bước thực hiện:
1. **Setup Environment** - Cài đặt dependencies
2. **Upload Project Files** - Upload code và data
3. **Configure API Keys** - Thiết lập các API keys
4. **Start API Server** - Khởi động server với Ngrok
5. **Test API** - Kiểm tra hoạt động

## 🔑 Yêu cầu:
- Google Gemini API Key
- Ngrok Auth Token (tùy chọn)
- Project files (services/, config/, data/)

## 1️⃣ Setup Environment

In [ ]:
# Cài đặt dependencies
!pip install fastapi uvicorn pyngrok python-dotenv
!pip install google-generativeai sentence-transformers transformers torch
!pip install qdrant-client accelerate tokenizers safetensors
!pip install psutil requests tqdm numpy

In [ ]:
# Kiểm tra GPU
import torch
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2️⃣ Upload Project Files

**Cách 1: Upload từ máy tính**
- Sử dụng file browser bên trái để upload
- Upload toàn bộ thư mục `rag_chatbot_hasaki`

**Cách 2: Clone từ Git (nếu có)**

In [ ]:
# Kiểm tra cấu trúc thư mục
import os

def show_directory_structure(path, prefix="", max_depth=3, current_depth=0):
    if current_depth > max_depth:
        return
    
    items = sorted(os.listdir(path))
    for i, item in enumerate(items):
        if item.startswith('.'):
            continue
            
        item_path = os.path.join(path, item)
        is_last = i == len(items) - 1
        
        current_prefix = "└── " if is_last else "├── "
        print(f"{prefix}{current_prefix}{item}")
        
        if os.path.isdir(item_path) and current_depth < max_depth:
            extension = "    " if is_last else "│   "
            show_directory_structure(item_path, prefix + extension, max_depth, current_depth + 1)

# Hiển thị cấu trúc project
project_path = "/content/rag_chatbot_hasaki"  # Điều chỉnh path nếu cần

if os.path.exists(project_path):
    print("📁 Cấu trúc project:")
    show_directory_structure(project_path)
else:
    print("❌ Chưa tìm thấy thư mục project. Vui lòng upload files!")
    print("💡 Hướng dẫn:")
    print("   1. Click vào folder icon bên trái")
    print("   2. Upload toàn bộ thư mục rag_chatbot_hasaki")
    print("   3. Chạy lại cell này để kiểm tra")

## 3️⃣ Configure API Keys

In [ ]:
# Thiết lập API keys
import os
from google.colab import userdata

# Cách 1: Sử dụng Colab Secrets (Khuyến nghị)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Đã lấy GEMINI_API_KEY từ Colab Secrets")
except:
    # Cách 2: Nhập trực tiếp (không an toàn)
    GEMINI_API_KEY = input("🔑 Nhập Gemini API Key: ")

# Ngrok Auth Token (tùy chọn)
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ Đã lấy NGROK_AUTH_TOKEN từ Colab Secrets")
except:
    NGROK_AUTH_TOKEN = input("🌐 Nhập Ngrok Auth Token (để trống nếu kh��ng có): ")
    if not NGROK_AUTH_TOKEN.strip():
        NGROK_AUTH_TOKEN = None

# Thiết lập environment variables
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
if NGROK_AUTH_TOKEN:
    os.environ['NGROK_AUTH_TOKEN'] = NGROK_AUTH_TOKEN

print("🔧 Đã thiết lập environment variables")

In [ ]:
# Tạo file .env cho project
project_path = "/content/rag_chatbot_hasaki"
env_content = f"""# API Keys
GEMINI_API_KEY={GEMINI_API_KEY}

# Qdrant Configuration
QDRANT_URL=http://localhost:6333
QDRANT_API_KEY=
QDRANT_COLLECTION_NAME=hasaki_products

# Model Configuration
EMBEDDING_MODEL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
RERANK_MODEL=BAAI/bge-reranker-base

# Generation Configuration
GEMINI_MODEL=gemini-1.5-flash
MAX_TOKENS=2048
TEMPERATURE=0.7
"""

env_path = os.path.join(project_path, ".env")
with open(env_path, "w", encoding="utf-8") as f:
    f.write(env_content)

print(f"✅ Đã tạo file .env tại: {env_path}")

## 4️⃣ Start API Server

In [ ]:
# Chuyển đến thư mục project
import os
os.chdir("/content/rag_chatbot_hasaki")
print(f"📂 Current directory: {os.getcwd()}")

In [ ]:
# Tạo file API Colab nếu chưa có
api_colab_code = '''#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# Code  api_colab.py sẽ được paste vào đây
# Hoặc copy từ file api_colab.py đã tạo
'''

# Kiểm tra file api_colab.py có tồn tại không
if not os.path.exists("api_colab.py"):
    print("⚠️ File api_colab.py không tồn tại!")
    print("💡 Vui lòng upload file api_colab.py vào thư mục project")
else:
    print("✅ File api_colab.py đã sẵn sàng")

In [ ]:
# Khởi động API server với Ngrok
import subprocess
import threading
import time

def run_api_server():
    """Chạy API server trong background"""
    cmd = ["python", "api_colab.py", "--port", "8000"]
    
    if NGROK_AUTH_TOKEN:
        cmd.extend(["--ngrok-token", NGROK_AUTH_TOKEN])
    
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )
    
    # In output real-time
    for line in process.stdout:
        print(line.strip())
        
        # Dừng sau khi server khởi động
        if "Uvicorn running" in line:
            break

# Chạy server trong thread riêng
print("🚀 Đang khởi động API server...")
server_thread = threading.Thread(target=run_api_server, daemon=True)
server_thread.start()

# Đợi server khởi động
time.sleep(10)
print("\n✅ API server đã được khởi động!")

## 5️⃣ Test API

In [ ]:
# Test API local
import requests
import json

def test_api(base_url="http://localhost:8000"):
    """Test các endpoint của API"""
    
    print(f"🔍 Testing API at: {base_url}")
    
    # Test health endpoint
    try:
        response = requests.get(f"{base_url}/health", timeout=10)
        if response.status_code == 200:
            data = response.json()
            print(f"✅ Health Check: {data['status']}")
            print(f"🤖 RAG Service: {'Ready' if data.get('rag_service_ready') else 'Not Ready'}")
        else:
            print(f"❌ Health Check Failed: {response.status_code}")
    except Exception as e:
        print(f"❌ Health Check Error: {e}")
        return False
    
    # Test system info
    try:
        response = requests.get(f"{base_url}/system-info", timeout=10)
        if response.status_code == 200:
            data = response.json()
            print(f"🖥️ Environment: {data.get('environment')}")
            if data.get('ngrok_url'):
                print(f"🌐 Ngrok URL: {data['ngrok_url']}")
                return data['ngrok_url']
    except Exception as e:
        print(f"⚠️ System Info Error: {e}")
    
    return True

# Test API
result = test_api()
if isinstance(result, str):  # Ngrok URL
    print(f"\n🎉 API đã sẵn sàng!")
    print(f"🔗 Public URL: {result}")
    print(f"📚 API Docs: {result}/docs")
elif result:
    print(f"\n🎉 API đã sẵn sàng tại localhost!")
    print(f"🔗 Local URL: http://localhost:8000")
else:
    print(f"\n❌ API chưa sẵn sàng. Kiểm tra lại!")

In [ ]:
# Test chat endpoint
def test_chat(api_url, message="Xin chào, bạn có thể giúp tôi tư vấn mỹ phẩm không?"):
    """Test chat functionality"""
    
    try:
        response = requests.post(
            f"{api_url}/chat",
            json={"message": message},
            timeout=30
        )
        
        if response.status_code == 200:
            data = response.json()
            print(f"✅ Chat Test Successful!")
            print(f"🙋 User: {message}")
            print(f"🤖 Bot: {data.get('answer', 'No answer')}")
            print(f"🎯 Intent: {data.get('intent', 'Unknown')}")
            print(f"⚡ Processing Time: {data.get('processing_time', 0):.2f}s")
            return True
        else:
            print(f"❌ Chat Test Failed: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        print(f"❌ Chat Test Error: {e}")
        return False

# Test với localhost trước
print("🧪 Testing Chat Functionality...")
test_chat("http://localhost:8000")

## 6️⃣ Get Public URLs

Copy các URL này để sử dụng trong Streamlit app:

In [ ]:
# Lấy tất cả URLs có sẵn
import requests

def get_all_urls():
    """Lấy tất cả URLs có sẵn"""
    
    urls = {
        "Local": "http://localhost:8000",
        "Ngrok": None
    }
    
    # Thử lấy Ngrok URL từ API
    try:
        response = requests.get("http://localhost:8000/system-info", timeout=5)
        if response.status_code == 200:
            data = response.json()
            if data.get('ngrok_url'):
                urls["Ngrok"] = data['ngrok_url']
    except:
        pass
    
    # Thử lấy từ pyngrok trực tiếp
    try:
        from pyngrok import ngrok
        tunnels = ngrok.get_tunnels()
        for tunnel in tunnels:
            if tunnel.config['addr'] == '8000':
                urls["Ngrok"] = tunnel.public_url
                break
    except:
        pass
    
    return urls

# Hiển thị URLs
urls = get_all_urls()

print("🔗 Available URLs:")
print("=" * 50)

for name, url in urls.items():
    if url:
        print(f"📍 {name}: {url}")
        print(f"   📚 Docs: {url}/docs")
        print(f"   🏥 Health: {url}/health")
        print()

# Highlight public URL
if urls["Ngrok"]:
    print("🌟 COPY THIS URL FOR STREAMLIT APP:")
    print(f"🔗 {urls['Ngrok']}")
    print()
    print("📋 Instructions:")
    print("1. Copy the Ngrok URL above")
    print("2. Paste it into your Streamlit app")
    print("3. Start chatting!")
else:
    print("⚠️ No public URL available. Using localhost only.")
    print("💡 Make sure Ngrok is properly configured.")

## 🎉 Deployment Complete!

### ✅ Những gì đã hoàn thành:
1. ✅ Cài đặt dependencies
2. ✅ Upload project files
3. ✅ Cấu hình API keys
4. ✅ Khởi động API server
5. ✅ Tạo Ngrok tunnel
6. ✅ Test API functionality

### 🔗 Sử dụng API:
- **Local URL**: `http://localhost:8000`
- **Public URL**: Copy từ cell trên
- **API Docs**: `{URL}/docs`
- **Health Check**: `{URL}/health`

### 📱 Kết nối Streamlit:
1. Chạy Streamlit app trên máy local hoặc cloud khác
2. Paste Public URL vào Streamlit interface
3. Test connection và bắt đầu chat!

### 💡 Lưu ý:
- Server sẽ chạy cho đến khi Colab session kết thúc
- Ngrok URL sẽ thay đổi mỗi lần restart
- Để server chạy liên tục, giữ tab Colab mở

---

**🌸 Happy Chatting with Hasaki RAG Chatbot! 🌸**